In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paper formal main-method worker

Thin engineering-canary entry. It uses an independent Drive tree and science_denominator=0; formal execution remains unauthorized.


In [ ]:
import json, os, pathlib, subprocess, sys
from google.colab import userdata

REPO = 'https://github.com/RICHAAARC/CEG-WM.git'
EXPECTED_EXACT = '93fc45a03ed3c15b1fde768316ba8db9dcff25e5'
JOB_ID = 'paper-main-engineering-canary-v1'
checkout = pathlib.Path('/content/cegwm-paper-formal')
runtime_root = pathlib.Path('/content/cegwm-paper-runtime/main')
drive_root = pathlib.Path('/content/drive/MyDrive/CEG-WM/PaperFormal-V1-EngineeringCanary')
if not checkout.exists(): subprocess.run(['git','clone',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
head=subprocess.run(['git','-C',str(checkout),'rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip()
dirty=subprocess.run(['git','-C',str(checkout),'status','--porcelain'],check=True,capture_output=True,text=True).stdout.strip()
assert head==EXPECTED_EXACT and not dirty
subprocess.run([sys.executable,'-m','pip','install','diffusers<0.40','transformers','accelerate','lpips','torchmetrics'],check=True)
child_env=dict(os.environ)
child_env['PYTHONPATH']=str(checkout/'src')+os.pathsep+str(checkout)
child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''
child_env['CEG_WM_ROOT_KEY']=userdata.get('CEG_WM_ROOT_KEY') or ''
assert child_env['HF_TOKEN'] and child_env['CEG_WM_ROOT_KEY']
command=[sys.executable,'-m','experiments.run_paper_main_worker','--job-id',JOB_ID,'--expected-exact',EXPECTED_EXACT,'--drive-root',str(drive_root/'main'),'--runtime-root',str(runtime_root),'--engineering-canary']
subprocess.run(command,cwd=checkout,env=child_env,check=True)
final_path=drive_root/'main'/JOB_ID/'canary_final.json'
public=json.loads(final_path.read_text(encoding='utf-8'))
print({'method_id':public['method_id'],'status':public['status'],'science_denominator':public['science_denominator'],'resume_verified':public['resume_verified']})
